# Chapter 18 — Giving the Agent Semantic Memory

Hands-On LangChain. Code targets **LangGraph 1.x** + **LangMem**.

Our baseline email assistant (Chapter 17) couldn't learn — every conversation started from scratch. Now
we give it **semantic memory**: the ability to store *facts* and retrieve them by *meaning*, not just
keywords. We'll add a memory **store** and two **LangMem tools** so the agent can save and search facts
about the user and their contacts, all in the **hot path** (as it responds).

Companion notebook for Chapter 18. Uses `prompts.py` (next to this notebook).

```bash
pip install langgraph langmem langchain langchain-openai python-dotenv -q
```

## 18.1 Setting the stage: the existing email agent

First we re-establish the foundation from Chapter 17 — the profile, triage rules, tools, and the triage
router. (This will look familiar; the memory parts come next.)

In [1]:
import os, warnings
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv(usecwd=True))
warnings.filterwarnings("ignore")

profile = {
    "name": "Youssef",
    "full_name": "Youssef Hosni",
    "user_profile_background": "Senior Data Scientist leading a team of 5 developers.",
}
prompt_instructions = {
    "triage_rules": {
        "ignore": "Marketing newsletters, spam emails, mass company announcements.",
        "notify": "Team member out sick, build system notifications, project status updates.",
        "respond": "Direct questions from team members, meeting requests, critical bug reports.",
    },
    "agent_instructions": "Use these tools when appropriate to help manage Youssef's tasks efficiently.",
}

In [2]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, Literal, Annotated
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

llm = init_chat_model(f"openai:{os.getenv('MODEL', 'gpt-5.4-mini')}")

class Router(BaseModel):
    """Analyze the unread email and route it according to its content."""
    reasoning: str = Field(description="Step-by-step reasoning behind the classification.")
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="'ignore', 'notify', or 'respond'.")
llm_router = llm.with_structured_output(Router)

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    return f"Email sent to {to} with subject '{subject}'"

@tool
def schedule_meeting(attendees: list[str], subject: str,
                     duration_minutes: int, preferred_day: str) -> str:
    """Schedule a calendar meeting."""
    return f"Meeting '{subject}' scheduled for {preferred_day} with {len(attendees)} attendees"

@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

## 18.2 Introducing tools for semantic memory

Semantic memory stores facts and retrieves them by meaning. We need two things: a **memory store** and
**tools** for the agent to use it.

### Setting up the memory store

We use LangGraph's `InMemoryStore`, configured with an **index** — an embedding model that converts each
memory into a vector. That's what makes the search *semantic* rather than keyword-based.

In [3]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore(
    index={"embed": f"openai:{os.getenv('EMBEDDING_MODEL', 'text-embedding-3-small')}"}
)

### Creating memory tools with LangMem

**LangMem** is a high-level library built on LangGraph that simplifies memory operations. It gives us
pre-configured tools to manage and search memory. The **`namespace`** organizes memories: a fixed app
bucket, a `{langgraph_user_id}` template filled at runtime (so different users' memories stay separate),
and a `collection` sub-folder.

In [4]:
import warnings; warnings.filterwarnings("ignore")
from langmem import create_manage_memory_tool, create_search_memory_tool

manage_memory_tool = create_manage_memory_tool(
    namespace=("email_assistant", "{langgraph_user_id}", "collection")
)
search_memory_tool = create_search_memory_tool(
    namespace=("email_assistant", "{langgraph_user_id}", "collection")
)

print("Manage tool:", manage_memory_tool.name)
print("Search tool:", search_memory_tool.name)

Manage tool: manage_memory
Search tool: search_memory


These are standard LangChain tools — `manage_memory` takes an action (create/update/delete) plus content, and `search_memory` takes a text query:

In [5]:
print(manage_memory_tool.description[:200])

Create, update, or delete a memory to persist across conversations.
Include the MEMORY ID when updating or deleting a MEMORY. Omit when creating a new MEMORY - it will be created for you.
Proactively 


## 18.3 Integrating memory into the agent

We update the system prompt to tell the agent about the two new tools, then build the response agent with
the expanded toolkit — and, crucially, pass it the **`store`**.

In [6]:
agent_system_prompt_memory = """
< Role >
You are {full_name}'s executive assistant. You are a top-notch executive assistant who cares about {name} performing as well as possible.
</ Role >
< Tools >
You have access to the following tools to help manage {name}'s communications and schedule:
1. write_email(to, subject, content) - Send emails to specified recipients
2. schedule_meeting(attendees, subject, duration_minutes, preferred_day) - Schedule calendar meetings
3. check_calendar_availability(day) - Check available time slots for a given day
4. manage_memory - Store any relevant information about contacts, actions, discussion, etc.
5. search_memory - Search for any relevant information that may have been stored in memory
</ Tools >
< Instructions >
{instructions}
</ Instructions >
"""

from langgraph.prebuilt import create_react_agent

def create_prompt(state):
    return [
        {"role": "system",
         "content": agent_system_prompt_memory.format(
             instructions=prompt_instructions["agent_instructions"], **profile)}
    ] + state["messages"]

tools = [write_email, schedule_meeting, check_calendar_availability,
         manage_memory_tool, search_memory_tool]

response_agent = create_react_agent(
    f"openai:{os.getenv('MODEL', 'gpt-5.4-mini')}",
    tools=tools,
    prompt=create_prompt,
    store=store,          # <-- gives the memory tools access to the store
)

## 18.4 Testing the memory in isolation

To use memory we provide a config with a **`langgraph_user_id`** — it gets slotted into the namespace
template. First, tell the agent a fact.

In [7]:
config = {"configurable": {"langgraph_user_id": "lance"}}

response = response_agent.invoke(
    {"messages": [{"role": "user", "content": "Jim is my friend"}]},
    config=config,
)
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Jim is my friend
================================== Ai Message ==================================
Tool Calls:
  manage_memory (call_RT2EEvAa63BoxyXPb5laKudZ)
 Call ID: call_RT2EEvAa63BoxyXPb5laKudZ
  Args:
    action: create
    content: Jim is Youssef's friend.
================================= Tool Message =================================
Name: manage_memory

created memory 76ccdde7-855d-4a02-9f2a-90a2ae5d0e94
================================== Ai Message ==================================

Got it — I’ll remember that Jim is your friend.


The agent called `manage_memory` to store the fact. Now, in a **separate invocation**, ask about Jim:

In [8]:
response = response_agent.invoke(
    {"messages": [{"role": "user", "content": "who is jim?"}]},
    config=config,
)
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

who is jim?
================================== Ai Message ==================================
Tool Calls:
  search_memory (call_T9Y9Zln9lKDS2AP5ZxGEMCB3)
 Call ID: call_T9Y9Zln9lKDS2AP5ZxGEMCB3
  Args:
    query: Jim
    limit: 5
================================= Tool Message =================================
Name: search_memory

[{"namespace":["email_assistant","lance","collection"],"key":"76ccdde7-855d-4a02-9f2a-90a2ae5d0e94","value":{"content":"Jim is Youssef's friend."},"created_at":"2026-06-21T20:04:48.581636+00:00","updated_at":"2026-06-21T20:04:48.581644+00:00","score":0.3463164209538834}]
================================== Ai Message ==================================

Jim is Youssef’s friend.


It used `search_memory` to recall the fact from a previous conversation — that's semantic memory working.

## 18.5 Building the final graph

Now we put the memory-enabled response agent back into the full email graph. The structure is the same
as Chapter 17 — a `triage_router` that routes to the response agent or ends — with one key change: we
pass the **`store`** to the whole graph at compile time.

In [9]:
from prompts import triage_system_prompt, triage_user_prompt
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.types import Command
from typing import Literal as TLiteral

class State(TypedDict):
    email_input: dict
    messages: Annotated[list, add_messages]

def triage_router(state: State) -> Command[TLiteral["response_agent", "__end__"]]:
    e = state["email_input"]
    system_prompt = triage_system_prompt.format(
        full_name=profile["full_name"], name=profile["name"], examples=None,
        user_profile_background=profile["user_profile_background"],
        triage_no=prompt_instructions["triage_rules"]["ignore"],
        triage_notify=prompt_instructions["triage_rules"]["notify"],
        triage_email=prompt_instructions["triage_rules"]["respond"],
    )
    user_prompt = triage_user_prompt.format(
        author=e["author"], to=e["to"], subject=e["subject"], email_thread=e["email_thread"],
    )
    result = llm_router.invoke([
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ])
    if result.classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {"messages": [{"role": "user", "content": f"Respond to the email {e}"}]}
    elif result.classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        goto, update = END, None
    else:
        print("🔔 Classification: NOTIFY - This email contains important information")
        goto, update = END, None
    return Command(goto=goto, update=update)

email_agent = StateGraph(State)
email_agent.add_node("triage_router", triage_router)
email_agent.add_node("response_agent", response_agent)
email_agent.add_edge(START, "triage_router")
email_agent = email_agent.compile(store=store)   # <-- store connected to the whole graph

## 18.6 A real-world scenario

Let's run a full email through the memory-enabled agent (with the same `config` so it shares the user's
memory namespace).

In [10]:
email_input = {
    "author": "Oliver Jack <Oliver.Jack@ToDataBeyond.com>",
    "to": "Youssef Hosni <Youssef.Hosni@ToDataBeyond.com>",
    "subject": "Quick question about API documentation",
    "email_thread": """Hi Youssef,

I was reviewing the API documentation for the new authentication service and noticed a few
endpoints seem to be missing. Could you clarify whether this was intentional?

Thanks!
Oliver""",
}
response = email_agent.invoke({"email_input": email_input}, config=config)
print(response["messages"][-1].content)

📧 Classification: RESPOND - This email requires a response


Done — I replied to Oliver confirming the missing endpoints were intentional and offered to share the supported/planned endpoints if needed.


The agent triaged the email, responded, and along the way could store facts about Oliver for future
interactions. Each time it handles mail for this user it builds up a richer picture, getting more
personalized over time.

We've given our assistant **semantic memory** — facts it stores and recalls in the hot path. In the next
chapter we add **episodic memory**: learning from past *examples* to improve how it triages email.